In [3]:
import torch
import os
from transformers import AutoTokenizer

os.chdir("..") # to allow relative imports
from model.roberta_encoder import *

torch.set_printoptions(linewidth=10000000, threshold=300000000)

/home/charles/.local/share/envs/sarformer/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/charles/.local/share/envs/sarformer/lib/python3.11/site-packages/torch/cuda/__init__.py:619: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/charles/.local/share/envs/sarformer/lib/python3.11/site-packages/torch/cuda/__init__.py:749: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at /opt/conda/conda-bld/pytorch_1716905969073/work/c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


In [4]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
assert tokenizer.is_fast


In [4]:
# this will need to go outside of the dataset/dataloader b/c it should
# receive batches of sentences
text = "Hello world"

tokenized_input = tokenizer(
    text,
    max_length=512,
    padding=False,
    truncation=True,
    return_tensors="pt",
)

x = tokenized_input.input_ids
mask = tokenized_input.attention_mask
x

tensor([[    0, 31414,   232,     2]])

In [5]:
roberta = RoBERTa(mask_proportion=.75)

In [7]:
masked_full_x, mask, x = roberta(x)


In [19]:
import pandas as pd
df = pd.read_csv("/data/million-case/all_data.csv", sep="@")

lens = []
for text in df["prompt"]:
    tokenized_input = tokenizer(
        str(text),
        max_length=512,
        padding=False,
        truncation=True,
        return_tensors="pt",
    )
    lens.append(tokenized_input.input_ids.shape[1])

In [23]:
from math import sqrt
from statistics import variance, mean, median

print("Min:", min(lens), "Max:", max(lens), "SD:", sqrt(variance(lens)), "E:", mean(lens), "Median:", median(lens))


Min: 3 Max: 512 SD: 50.437237357152725 E: 107.7812228882791 Median: 98.0
